# 🌐 AI Multi-Language Translator — Gradio (Colab)
**CodeAlpha Task — Python + Gradio + Translation API**

A professional, attractive web-based GUI built with Gradio Blocks. Runs
natively inside Colab — no tunnel setup needed. Gradio renders the UI
**inline in this notebook** and also gives you a public shareable link.

**Features**
- Modern card-style layout (Gradio Blocks + custom CSS)
- Source / target language dropdowns (16 languages incl. Auto Detect source)
- 🌍 Translate button, plus an optional ⚡ **Live translate as you type** toggle
- ⇄ Swap languages
- Built-in copy-to-clipboard on both text boxes
- 🔊 Listen — inline autoplaying text-to-speech (gTTS)
- 🗑 Clear button
- Clickable translation history table

Run the cells below in order: **Runtime → Run all**.


In [ ]:
# 1. Install dependencies
!pip install -q gradio deep-translator gTTS


In [ ]:
# 2. Imports, config, and core translation/TTS logic
import tempfile
from datetime import datetime

import gradio as gr
from deep_translator import GoogleTranslator
from gtts import gTTS

LANGUAGES = {
    "Auto Detect": ("auto", None),
    "English": ("en", "en"),
    "Urdu": ("ur", "ur"),
    "Arabic": ("ar", "ar"),
    "French": ("fr", "fr"),
    "Spanish": ("es", "es"),
    "German": ("de", "de"),
    "Chinese (Simplified)": ("zh-CN", "zh-CN"),
    "Hindi": ("hi", "hi"),
    "Turkish": ("tr", "tr"),
    "Japanese": ("ja", "ja"),
    "Russian": ("ru", "ru"),
    "Portuguese": ("pt", "pt"),
    "Italian": ("it", "it"),
    "Bengali": ("bn", "bn"),
    "Persian": ("fa", "fa"),
}
TARGET_LANGUAGES = {k: v for k, v in LANGUAGES.items() if k != "Auto Detect"}
HISTORY_HEADERS = ["Time", "From", "To", "Source Text", "Translation"]
MAX_HISTORY = 50

CUSTOM_CSS = """
#header-title { text-align: center; margin-bottom: 0; }
#header-sub { text-align: center; color: gray; margin-top: 0; }
.status-ready { color: gray; }
.status-ok { color: #1e8e3e; font-weight: 600; }
.status-busy { color: #1a73e8; font-weight: 600; }
.status-error { color: #d93025; font-weight: 600; }
footer { visibility: hidden }
"""


def status_html(text, kind="ready"):
    cls = {"ready": "status-ready", "ok": "status-ok", "busy": "status-busy", "error": "status-error"}[kind]
    return f"<span class=\'{cls}\'>{text}</span>"


def history_to_rows(history):
    return [[h["time"], h["src_lang"], h["tgt_lang"], h["src_text"], h["tgt_text"]] for h in history]


def translate_core(text, source_lang, target_lang, history):
    text = (text or "").strip()
    if not text:
        return "", status_html("Ready"), history, history_to_rows(history)

    src_code = LANGUAGES[source_lang][0]
    tgt_code = TARGET_LANGUAGES[target_lang][0]

    try:
        result = GoogleTranslator(source=src_code, target=tgt_code).translate(text)
    except Exception as e:
        return "", status_html(f"Translation failed: {e}", "error"), history, history_to_rows(history)

    new_entry = {
        "time": datetime.now().strftime("%H:%M:%S"),
        "src_lang": source_lang,
        "tgt_lang": target_lang,
        "src_text": text,
        "tgt_text": result,
    }
    history = [new_entry] + history
    history = history[:MAX_HISTORY]

    return result, status_html("\u2713 Translated", "ok"), history, history_to_rows(history)


def on_translate_click(text, source_lang, target_lang, history):
    return translate_core(text, source_lang, target_lang, history)


def on_live_change(text, source_lang, target_lang, live_enabled, history, current_output, current_status):
    if not live_enabled:
        return current_output, current_status, history, history_to_rows(history)
    return translate_core(text, source_lang, target_lang, history)


def on_clear():
    return "", "", status_html("Ready"), None, None


def on_swap(source_lang, target_lang, input_text, output_text):
    if source_lang == "Auto Detect":
        return source_lang, target_lang, input_text, output_text, status_html("Can\'t swap from Auto Detect", "error")
    return target_lang, source_lang, output_text, input_text, status_html("Languages swapped", "ok")


def make_speech(text, lang_code):
    text = (text or "").strip()
    if not text:
        return None, status_html("Nothing to read aloud", "error")
    if lang_code is None:
        return None, status_html("Text-to-speech isn\'t available for Auto Detect", "error")
    try:
        tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        gTTS(text=text, lang=lang_code).save(tmp.name)
        return tmp.name, status_html("\u2713 Playing audio", "ok")
    except Exception as e:
        return None, status_html(f"Audio error: {e}", "error")


def on_listen_input(text, source_lang):
    return make_speech(text, LANGUAGES[source_lang][1])


def on_listen_output(text, target_lang):
    return make_speech(text, TARGET_LANGUAGES[target_lang][1])


def on_clear_history():
    return [], []


def on_history_select(history, evt: gr.SelectData):
    if evt.index is None or not history:
        return gr.update(), gr.update(), gr.update(), gr.update(), status_html("Ready")
    row_idx = evt.index[0] if isinstance(evt.index, (list, tuple)) else evt.index
    if row_idx >= len(history):
        return gr.update(), gr.update(), gr.update(), gr.update(), status_html("Ready")
    item = history[row_idx]
    return item["src_lang"], item["tgt_lang"], item["src_text"], item["tgt_text"], status_html("Loaded from history", "ok")

print("Core logic loaded.")


In [ ]:
# 3. Build the Gradio UI (Blocks)

with gr.Blocks(title="AI Translator") as demo:

    history_state = gr.State([])

    gr.Markdown("# \U0001f310 AI Translator", elem_id="header-title")
    gr.Markdown("Professional real-time translation, powered by an online translation API", elem_id="header-sub")

    with gr.Row():
        source_dd = gr.Dropdown(choices=list(LANGUAGES.keys()), value="English", label="Source Language", scale=5)
        swap_btn = gr.Button("\u21c4", scale=1, min_width=60)
        target_dd = gr.Dropdown(choices=list(TARGET_LANGUAGES.keys()), value="Urdu", label="Target Language", scale=5)

    live_toggle = gr.Checkbox(label="\u26a1 Live translate as you type (no button needed)", value=False)

    with gr.Row():
        with gr.Column():
            input_box = gr.Textbox(
                label="Enter text", lines=7, placeholder="Type or paste text here...",
                buttons=["copy"],
            )
            with gr.Row():
                translate_btn = gr.Button("\U0001f30d Translate", variant="primary")
                listen_in_btn = gr.Button("\U0001f50a Listen")
                clear_btn = gr.Button("\U0001f5d1 Clear")
            input_audio = gr.Audio(label="Input Audio", autoplay=True, visible=True)

        with gr.Column():
            output_box = gr.Textbox(
                label="Translation", lines=7, interactive=False, buttons=["copy"],
            )
            with gr.Row():
                listen_out_btn = gr.Button("\U0001f50a Listen")
            output_audio = gr.Audio(label="Result Audio", autoplay=True, visible=True)

    status_md = gr.HTML(status_html("Ready"))

    with gr.Accordion("\U0001f4dc Translation History", open=False):
        clear_history_btn = gr.Button("Clear History", size="sm")
        history_table = gr.Dataframe(
            headers=HISTORY_HEADERS, datatype=["str"] * 5, interactive=False,
            wrap=True, row_count=(0, "dynamic"),
        )
        gr.Markdown("*Click any row above to reload that translation.*")

    # ---- Wiring ----

    translate_btn.click(
        fn=on_translate_click,
        inputs=[input_box, source_dd, target_dd, history_state],
        outputs=[output_box, status_md, history_state, history_table],
    )
    input_box.submit(
        fn=on_translate_click,
        inputs=[input_box, source_dd, target_dd, history_state],
        outputs=[output_box, status_md, history_state, history_table],
    )
    input_box.change(
        fn=on_live_change,
        inputs=[input_box, source_dd, target_dd, live_toggle, history_state, output_box, status_md],
        outputs=[output_box, status_md, history_state, history_table],
    )

    swap_btn.click(
        fn=on_swap,
        inputs=[source_dd, target_dd, input_box, output_box],
        outputs=[source_dd, target_dd, input_box, output_box, status_md],
    )

    clear_btn.click(
        fn=on_clear,
        outputs=[input_box, output_box, status_md, input_audio, output_audio],
    )

    listen_in_btn.click(
        fn=on_listen_input, inputs=[input_box, source_dd], outputs=[input_audio, status_md],
    )
    listen_out_btn.click(
        fn=on_listen_output, inputs=[output_box, target_dd], outputs=[output_audio, status_md],
    )

    clear_history_btn.click(fn=on_clear_history, outputs=[history_state, history_table])

    history_table.select(
        fn=on_history_select,
        inputs=[history_state],
        outputs=[source_dd, target_dd, input_box, output_box, status_md],
    )

print("UI built.")


## 4. Launch

Running the cell below starts the app **inline in this notebook** and also
prints a public `*.gradio.live` link (valid for 72 hours) you can open on
your phone or share with someone else.

If you only want the inline view (no public link), change `share=True` to
`share=False`.


In [ ]:
# 5. Launch the app
demo.launch(share=True, theme=gr.themes.Soft(primary_hue="blue", secondary_hue="sky"), css=CUSTOM_CSS, debug=False)
